# 1. 觸發與備份：
邊緣層的 Load Balancer / Gateway (Raspberry Pi) 定時執行 backup.sh，將 HORNET 帳本資料打包加密成 .enc 檔。

# 2. 傳輸與路由：
Gateway 透過 API 將 .enc 備份檔傳給 後端 Backend (API Gateway / Background Worker)。

# 3. 儲存與歸檔：
Background Worker（非同步背景任務）將這個大檔案寫入右下角的 【物件/日誌儲存 Local Vault / S3】。
同時，將「備份時間、備份檔名、檔案大小」等文字紀錄寫入 【主資料庫 PostgreSQL / MySQL】 供 Admin 查詢備份清單。

# 4. 災難還原 (Restore)：
當新 Gateway 啟動時，向後端發出請求，後端從 【物件/日誌儲存 Local Vault / S3】 抓取最新的 .enc 備份檔發還給 Gateway 進行解密復原。

# backup.sh

In [ ]:
#!/bin/bash

# Set variables (modify according to the actual path.)
BACKUP_DATE=$(date +%Y%m%d_%H%M%S)
BACKUP_DIR="backup_house_A_${BACKUP_DATE}"
SECRET_KEY="AdminSecretPassword123"  # AES-256 Encrypted password
HORNET_DATA_PATH="/home/iota/iota-private/one-command-tangle"  #HORNET Database path

echo "=== Start the backup job [$BACKUP_DATE] ==="

mkdir -p /tmp/${BACKUP_DIR}   # Create a temporary work folder
echo "1. Copy HORNET ledger data and files"
cp -r ${HORNET_DATA_PATH}/db /tmp/${BACKUP_DIR}/
cp -r ${HORNET_DATA_PATH}/config /tmp/${BACKUP_DIR}/
echo "2. Package and encrypt using AES-256"
tar -czf - -C /tmp/${BACKUP_DIR} . | openssl enc -aes-256-cbc -pbkdf2 -k ${SECRET_KEY} -out /tmp/${BACKUP_DIR}.enc

# Call the script or use the curl API to upload to the cloud server
curl -X POST -F "file=@/tmp/${BACKUP_DIR}.enc" https://sv1.alexc.one/api/v1/backups/upload
# Clean up temporary unencrypted files
rm -rf /tmp/${BACKUP_DIR} /tmp/${BACKUP_DIR}.enc

echo "=== Backup successful. The encrypted backup file has been stored in /tmp/${BACKUP_DIR}.enc ==="

# 排程編輯
每天凌晨兩點，自動執行backup.sh(備份)

In [ ]:
crontab -e

0 2 * * 0 /home/iota/backup.sh >> /tmp/backup.log 2>&1

# restore.sh

In [ ]:
#!/bin/bash

# step 1: Check parameters: Must specify which '.enc' backup file to restore
if [ -z "$1" ]; then
    echo "Error: Please specify the path to the encrypted backup file to restore!"
    echo "Example: ./restore.sh /tmp/backup_house_A_20260807_150031.enc"
    exit 1
fi

ENC_FILE="$1"
SECRET_KEY="AdminSecretPassword123"  # Must be exactly the same as "backup.sh"'s password
TANGLE_PATH="/home/iota/iota-private/one-command-tangle"
TEMP_RESTORE_DIR="/tmp/restore_temp"
CONTAINER_NAME="one-command-tangle_iri_1"  # Docker container name

# Check if the backup file exists
if [ ! -f "${ENC_FILE}" ]; then
    echo "Error: Backup file ${ENC_FILE} not found"
    exit 1
fi

echo "=== Start the restore operation ==="
echo "Expected file to be restored: ${ENC_FILE}"

# step 2: Pause the running Docker container
echo "Stopping the HORNET node container ..."
sudo docker stop ${CONTAINER_NAME} 2>/dev/null || true

# step 3: Create a temporary folder for decryption
mkdir -p ${TEMP_RESTORE_DIR}

# step 4: Perform AES-256 decryption & decompression
echo "Performing AES-256 decryption & decompression ..."
openssl enc -d -aes-256-cbc -pbkdf2 -k ${SECRET_KEY} -in ${ENC_FILE} | tar -xzf - -C ${TEMP_RESTORE_DIR}

# step 5: Clear corrupt/old data and overwrite with backup data
echo "Overwiting and restoring Tangle ledger DB & Config ..."
sudo rm -rf ${TANGLE_PATH}/db ${TANGLE_PATH}/config
sudo cp -r ${TEMP_RESTORE_DIR}/db ${TANGLE_PATH}/
sudo cp -r ${TEMP_RESTORE_DIR}/config ${TANGLE_PATH}/

# step 6: Clean up temporary decryption folder
rm -rf ${TEMP_RESTORE_DIR}

# step 7: Restart Docker container
echo "Restarting HORNET node container ..."
sudo docker start ${CONTAINER_NAME} 2>/dev/null || true

echo "=== Restoration successful! Ledger data has been fully recovered and reconnected ==="